In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier

from sklearn.model_selection import train_test_split, KFold

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    f1_score,
    hamming_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
    auc,
    multilabel_confusion_matrix
)

from transformers import BertTokenizer, BertModel

import matplotlib.pyplot as plt
import seaborn as sns


## Primary Classification

In [ ]:
# Import data here
# 

In [ ]:
# Primary classification

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fpr_list = {label: [] for label in label_classes_new} 
tpr_list = {label: [] for label in label_classes_new} 
prec_list = {label: [] for label in label_classes_new} 
rec_list = {label: [] for label in label_classes_new} 
auc_list = {label: [] for label in label_classes_new} 

f1_micro_tuned = []
f1_macro_tuned = []
auc_scores = []
hamming_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(text_notes, stratify_labels), 1):

    X_train_text, X_test_text = np.array(text_notes)[train_idx], np.array(text_notes)[test_idx]
    y_train, y_test = y_full_combined[train_idx], y_full_combined[test_idx]

    vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1,2))
    X_train = vectorizer.fit_transform(X_train_text)
    X_test  = vectorizer.transform(X_test_text)

    classifier = OneVsRestClassifier(RandomForestClassifier(random_state=42))
    classifier.fit(X_train, y_train)

    y_proba = classifier.predict_proba(X_test)
    y_pred_default = classifier.predict(X_test)

    # optimize threshold per class
    thresholds_opt = {}
    predictions_optimized = np.zeros_like(y_test)

    for i, label in enumerate(label_classes_new):
        class_true = y_test[:, i]
        class_score = y_proba[:, i]

        # maximize f1
        thresholds_to_try = np.linspace(0.1, 0.9, 81)
        best_f1 = 0
        best_thresh = 0.5

        for thresh in thresholds_to_try:
            class_pred = (class_score >= thresh).astype(int)
            f1 = f1_score(class_true, class_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh

        thresholds_opt[label] = best_thresh
        predictions_optimized[:, i] = (class_score >= best_thresh).astype(int)


        class_auc = roc_auc_score(class_true, class_score)
        auc_list[label].append(class_auc)

        precision, recall, _ = precision_recall_curve(class_true, class_score)
        fpr, tpr, _ = roc_curve(class_true, class_score)

        prec_list[label].append(precision)
        rec_list[label].append(recall)
        fpr_list[label].append(fpr)
        tpr_list[label].append(tpr)


    
    f1_micro_tuned.append(f1_score(y_test, predictions_optimized, average='micro'))
    f1_macro_tuned.append(f1_score(y_test, predictions_optimized, average='macro'))
    auc_scores.append(roc_auc_score(y_test, y_proba, average='macro'))
    hamming_scores.append(hamming_loss(y_test, predictions_optimized))


print("AUC: ", np.nanmean(auc_scores), "±", np.nanstd(auc_scores))
print("F1 micro: ", np.nanmean(f1_micro_tuned), "±", np.nanstd(f1_micro_tuned))
print("F1 macro: ", np.nanmean(f1_macro_tuned), "±", np.nanstd(f1_macro_tuned))
print("Hamming Loss: ", np.nanmean(hamming_scores), "±", np.nanstd(hamming_scores))

## Seconday Classification

In [ ]:
def secondary_classifier_multi(notes, labels, synth_notes=None, synth_labels=None,
                  label_classes=None, pos_label=None, n_splits=3):
    
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fpr_list = {i: [] for i in range(len(label_classes))}
    tpr_list = {i: [] for i in range(len(label_classes))}
    prec_list = {i: [] for i in range(len(label_classes))}
    rec_list = {i: [] for i in range(len(label_classes))}

    acc_tuned = [] 
    f1_micro_tuned = []
    f1_macro_tuned = [] 
    hamming_tuned = []
    auc_list = []
    auprc_list = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(notes, labels), 1):
        
        x_train, x_test = np.array(notes)[train_idx], np.array(notes)[test_idx]
        y_train, y_test = np.array(labels)[train_idx], np.array(labels)[test_idx]

        x_train_all = list(synth_notes) + list(x_train)
        y_train_all = np.concatenate([synth_labels, y_train])

        vect = TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english', min_df=2)
        X_train = vect.fit_transform(x_train_all)
        X_test = vect.transform(x_test)

        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train_all)

        y_pred_default = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        # get threshold to optimize F1 macro
        best_thresh = 0
        best_f1 = 0
        y_pred_tuned = y_pred_default.copy()
        fallback = np.bincount(y_test).argmax()

        for t in np.arange(0.0, 1.0, 0.05):
            max_probs = np.max(y_proba, axis=1)
            preds = np.array([label_classes[i] for i in np.argmax(y_proba, axis=1)])
            if t > 0:
                preds[max_probs < t] = fallback
            f1 = f1_score(y_test, preds, average='macro', 
                         labels=label_classes, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thresh = f1, t
                y_pred_tuned = preds.copy()

        acc_tuned.append(accuracy_score(y_test, y_pred_tuned))
        f1_micro_tuned.append(f1_score(y_test, y_pred_tuned, average='micro', 
                                                   labels=label_classes, zero_division=0))
        f1_macro_tuned.append(f1_score(y_test, y_pred_tuned, average='macro', 
                                                   labels=label_classes, zero_division=0))
        hamming_tuned.append(hamming_loss(y_test, y_pred_tuned))


        y_test_bin = label_binarize(y_test, classes=label_classes)
        auc_list.append(roc_auc_score(y_test_bin, y_proba, 
                                            average='macro', multi_class='ovr'))
        
        
        
        auprcs = []
        for i in range(len(label_classes)):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
            prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_proba[:, i])
            auprcs.append(auc(rec, prec))
            #save and return these for later plotting
            fpr_list[i].append(fpr)
            tpr_list[i].append(tpr)
            prec_list[i].append(prec)
            rec_list[i].append(rec)
        auprc_list.append(np.mean(auprcs))

    print("AUC: ", np.nanmean(auc_list), "±", np.nanstd(auc_list))
    print("AUPRC: ",np.nanmean(auprc_list), "±", np.nanstd(auprc_list))
    print("F1 micro: ", np.nanmean(f1_micro_tuned), "±", np.nanstd(f1_micro_tuned))
    print("F1 macro: ", np.nanmean(f1_macro_tuned), "±", np.nanstd(f1_macro_tuned))
    print("Hamming Loss: ", np.nanmean(hamming_tuned), "±", np.nanstd(hamming_tuned))

    return fpr_list, tpr_list, prec_list, rec_list
        